# DogHeart X-ray Classification Homework

**????**?2025120004 ???  
**??**?DogHeart ????????????Large / Normal / Small?

? notebook ??????????????? PyTorch ??????????????????????? GitHub?

- GitHub ???[https://github.com/wang935/homework](https://github.com/wang935/homework)
- ???? `.pt`?[https://github.com/wang935/homework/blob/main/models/final_model.pt](https://github.com/wang935/homework/blob/main/models/final_model.pt)
- ???? CSV?[https://github.com/wang935/homework/blob/main/outputs/results.csv](https://github.com/wang935/homework/blob/main/outputs/results.csv)
- ???? JSON?[https://github.com/wang935/homework/blob/main/outputs/convnext_tiny_final/valid_metrics.json](https://github.com/wang935/homework/blob/main/outputs/convnext_tiny_final/valid_metrics.json)
- ?? PDF?[https://github.com/wang935/homework/blob/main/report/main_final.pdf](https://github.com/wang935/homework/blob/main/report/main_final.pdf)


## 1. Build your own convolutional neural network using PyTorch

?????? `src/` ? `scripts/`???????????????????? `DogHeartCNN`??????????? ImageNet ??? ConvNeXt Tiny??????? `architecture=convnext_tiny`?


In [ ]:
from __future__ import annotations

from collections.abc import Callable
from dataclasses import dataclass
from typing import Any

import torch
from torch import nn
from torchvision import models


class ConvBNAct(nn.Sequential):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1) -> None:
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
        )


class ResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, dropout: float = 0.0) -> None:
        super().__init__()
        self.conv1 = ConvBNAct(in_channels, out_channels, stride=stride)
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.act = nn.SiLU(inplace=True)
        self.drop = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.shortcut(x)
        x = self.conv1(x)
        x = self.drop(x)
        x = self.conv2(x)
        return self.act(x + residual)


class DogHeartCNN(nn.Module):
    """A compact custom CNN for dog chest X-ray heart-size classification."""

    def __init__(self, num_classes: int = 3, dropout: float = 0.25) -> None:
        super().__init__()
        self.features = nn.Sequential(
            ConvBNAct(3, 32, stride=2),
            ConvBNAct(32, 48),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            ResidualBlock(48, 64, stride=1, dropout=0.05),
            ResidualBlock(64, 96, stride=2, dropout=0.08),
            ResidualBlock(96, 128, stride=1, dropout=0.08),
            ResidualBlock(128, 192, stride=2, dropout=0.10),
            ResidualBlock(192, 256, stride=1, dropout=0.10),
            ResidualBlock(256, 320, stride=2, dropout=0.12),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(320, 128),
            nn.SiLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x))


@dataclass(frozen=True)
class TorchvisionModelSpec:
    builder: Callable[..., nn.Module]
    default_weights: Any
    replace_head: Callable[[nn.Module, int, float], None]


def _dropout_classifier(in_features: int, num_classes: int, dropout: float) -> nn.Sequential:
    return nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_features, num_classes),
    )


def _replace_fc_with_dropout(model: nn.Module, num_classes: int, dropout: float) -> None:
    in_features = model.fc.in_features
    model.fc = _dropout_classifier(in_features, num_classes, dropout)


def _replace_fc(model: nn.Module, num_classes: int, dropout: float) -> None:
    del dropout
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)


def _replace_classifier_with_dropout(model: nn.Module, num_classes: int, dropout: float) -> None:
    in_features = model.classifier.in_features
    model.classifier = _dropout_classifier(in_features, num_classes, dropout)


def _replace_classifier_tail_with_dropout(model: nn.Module, num_classes: int, dropout: float) -> None:
    in_features = model.classifier[-1].in_features
    model.classifier = _dropout_classifier(in_features, num_classes, dropout)


def _replace_classifier_tail(model: nn.Module, num_classes: int, dropout: float) -> None:
    del dropout
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)


def _replace_head(model: nn.Module, num_classes: int, dropout: float) -> None:
    del dropout
    in_features = model.head.in_features
    model.head = nn.Linear(in_features, num_classes)


TORCHVISION_MODEL_SPECS: dict[str, TorchvisionModelSpec] = {
    "resnet18": TorchvisionModelSpec(models.resnet18, models.ResNet18_Weights.DEFAULT, _replace_fc_with_dropout),
    "resnet34": TorchvisionModelSpec(models.resnet34, models.ResNet34_Weights.DEFAULT, _replace_fc_with_dropout),
    "resnet50": TorchvisionModelSpec(models.resnet50, models.ResNet50_Weights.DEFAULT, _replace_fc_with_dropout),
    "densenet121": TorchvisionModelSpec(
        models.densenet121,
        models.DenseNet121_Weights.DEFAULT,
        _replace_classifier_with_dropout,
    ),
    "efficientnet_b0": TorchvisionModelSpec(
        models.efficientnet_b0,
        models.EfficientNet_B0_Weights.DEFAULT,
        _replace_classifier_tail_with_dropout,
    ),
    "efficientnet_b2": TorchvisionModelSpec(
        models.efficientnet_b2,
        models.EfficientNet_B2_Weights.DEFAULT,
        _replace_classifier_tail_with_dropout,
    ),
    "efficientnet_v2_s": TorchvisionModelSpec(
        models.efficientnet_v2_s,
        models.EfficientNet_V2_S_Weights.DEFAULT,
        _replace_classifier_tail_with_dropout,
    ),
    "convnext_tiny": TorchvisionModelSpec(
        models.convnext_tiny,
        models.ConvNeXt_Tiny_Weights.DEFAULT,
        _replace_classifier_tail,
    ),
    "convnext_small": TorchvisionModelSpec(
        models.convnext_small,
        models.ConvNeXt_Small_Weights.DEFAULT,
        _replace_classifier_tail,
    ),
    "mobilenet_v3_large": TorchvisionModelSpec(
        models.mobilenet_v3_large,
        models.MobileNet_V3_Large_Weights.DEFAULT,
        _replace_classifier_tail,
    ),
    "regnet_y_400mf": TorchvisionModelSpec(
        models.regnet_y_400mf,
        models.RegNet_Y_400MF_Weights.DEFAULT,
        _replace_fc,
    ),
    "regnet_y_800mf": TorchvisionModelSpec(
        models.regnet_y_800mf,
        models.RegNet_Y_800MF_Weights.DEFAULT,
        _replace_fc,
    ),
    "swin_t": TorchvisionModelSpec(models.swin_t, models.Swin_T_Weights.DEFAULT, _replace_head),
}

SUPPORTED_ARCHITECTURES: tuple[str, ...] = ("custom", *TORCHVISION_MODEL_SPECS.keys())


def _resolve_weights(default_weights: Any, pretrained: bool) -> Any:
    return default_weights if pretrained else None


def build_model(
    num_classes: int = 3,
    dropout: float = 0.25,
    architecture: str = "custom",
    pretrained: bool = False,
) -> nn.Module:
    architecture = architecture.lower()
    if architecture == "custom":
        return DogHeartCNN(num_classes=num_classes, dropout=dropout)

    spec = TORCHVISION_MODEL_SPECS.get(architecture)
    if spec is None:
        supported = ", ".join(SUPPORTED_ARCHITECTURES)
        raise ValueError(f"Unknown architecture: {architecture}. Supported architectures: {supported}")

    model = spec.builder(weights=_resolve_weights(spec.default_weights, pretrained))
    spec.replace_head(model, num_classes, dropout)
    return model


## 2. Data loading and preprocessing

???????? `torchvision.datasets.ImageFolder`????????? `Large=0, Normal=1, Small=2`??????????? CSV ??????


In [ ]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Callable

from PIL import Image, ImageOps
import torch
from torch.utils.data import Dataset
from torchvision import datasets, transforms


CLASS_NAMES = ["Large", "Normal", "Small"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
ImageSize = int | tuple[int, int]


def natural_key(path: Path) -> list[object]:
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", path.name)]


class TestImageDataset(Dataset):
    """Test dataset returning image tensor and original file name."""

    def __init__(self, image_dir: str | Path, transform: Callable | None = None) -> None:
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.paths = sorted(
            [p for p in self.image_dir.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"}],
            key=natural_key,
        )
        if not self.paths:
            raise FileNotFoundError(f"No images found in {self.image_dir}")

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, str]:
        path = self.paths[index]
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, path.name


class ResizePad:
    def __init__(self, size: ImageSize, fill: int = 0) -> None:
        self.size = _resize_size(size)
        self.fill = fill

    def __call__(self, image: Image.Image) -> Image.Image:
        target_h, target_w = self.size
        image = ImageOps.contain(image, (target_w, target_h), method=Image.Resampling.BILINEAR)
        canvas = Image.new(image.mode, (target_w, target_h), color=self.fill)
        left = (target_w - image.width) // 2
        top = (target_h - image.height) // 2
        canvas.paste(image, (left, top))
        return canvas


def _resize_size(image_size: ImageSize) -> tuple[int, int]:
    return image_size if isinstance(image_size, tuple) else (image_size, image_size)


def _resize_transform(image_size: ImageSize, resize_mode: str) -> transforms.Resize | ResizePad:
    if resize_mode == "stretch":
        return transforms.Resize(_resize_size(image_size))
    if resize_mode == "pad":
        return ResizePad(image_size)
    raise ValueError(f"Unknown resize mode: {resize_mode}")


def train_transform(image_size: ImageSize = 224, strength: str = "standard", resize_mode: str = "stretch") -> transforms.Compose:
    normalize = transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    resize = _resize_transform(image_size, resize_mode)
    if strength == "none":
        return eval_transform(image_size, resize_mode)
    if strength == "light":
        return transforms.Compose(
            [
                resize,
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(degrees=3),
                transforms.ColorJitter(brightness=0.05, contrast=0.05),
                transforms.ToTensor(),
                normalize,
            ]
        )
    if strength == "randaug":
        return transforms.Compose(
            [
                resize,
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandAugment(num_ops=2, magnitude=9),
                transforms.ToTensor(),
                normalize,
            ]
        )
    if strength == "strong":
        return transforms.Compose(
            [
                resize,
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(degrees=10),
                transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.05),
                transforms.RandAugment(num_ops=2, magnitude=7),
                transforms.ToTensor(),
                normalize,
            ]
        )
    if strength != "standard":
        raise ValueError(f"Unknown augmentation strength: {strength}")
    return transforms.Compose(
        [
            resize,
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=8),
            transforms.ColorJitter(brightness=0.12, contrast=0.12),
            transforms.ToTensor(),
            normalize,
        ]
    )


def eval_transform(image_size: ImageSize = 224, resize_mode: str = "stretch") -> transforms.Compose:
    return transforms.Compose(
        [
            _resize_transform(image_size, resize_mode),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ]
    )


def make_imagefolder(root: str | Path, transform: Callable | None = None) -> datasets.ImageFolder:
    dataset = datasets.ImageFolder(str(root), transform=transform)
    if dataset.classes != CLASS_NAMES:
        raise ValueError(
            f"Unexpected class order {dataset.classes}. Expected {CLASS_NAMES}; "
            "the output CSV depends on this fixed mapping."
        )
    return dataset


## 3. Train and evaluate the model

?????????????? checkpoint metadata ??

- architecture: `convnext_tiny`
- pretrained: `True`
- image size: `224 x 224`
- augmentation: `strong`
- MixUp alpha: `0.4`
- label smoothing: `0.1`
- batch size: `48`
- learning rate: `2e-4`
- weight decay: `0.1`
- best epoch: `23`

????????????????????????


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO_ROOT = Path(r"C:\Users\wang\Desktop\Homework")
PYTHON = REPO_ROOT / ".venv" / "Scripts" / "python.exe"
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

print("Repository:", REPO_ROOT)
print("Python:", PYTHON)


In [ ]:
# Training command used for the final recipe.
# This is left as a reproducible command; running it retrains the model.
train_cmd = [
    str(PYTHON), "scripts/train.py",
    "--architecture", "convnext_tiny",
    "--pretrained",
    "--image-size", "224",
    "--aug-strength", "strong",
    "--mixup-alpha", "0.4",
    "--epochs", "80",
    "--batch-size", "48",
    "--lr", "2e-4",
    "--weight-decay", "0.1",
    "--model-dir", "models",
    "--output-dir", "outputs/convnext_tiny_final",
]
print(" ".join(train_cmd))
# subprocess.run(train_cmd, cwd=REPO_ROOT, check=True)


In [ ]:
# Re-evaluate the uploaded final checkpoint on the validation split.
eval_cmd = [
    str(PYTHON), "scripts/evaluate.py",
    "--checkpoint", "models/final_model.pt",
    "--output", "outputs/convnext_tiny_final/valid_metrics_recheck.json",
    "--batch-size", "64",
    "--num-workers", "0",
]
subprocess.run(eval_cmd, cwd=REPO_ROOT, check=True)


In [ ]:
# Generate and validate the no-header test prediction CSV.
predict_cmd = [
    str(PYTHON), "scripts/predict.py",
    "--checkpoint", "models/final_model.pt",
    "--output", "outputs/results.csv",
    "--batch-size", "64",
]
check_cmd = [str(PYTHON), "scripts/check_submission.py", "--csv", "outputs/results.csv"]

# Uncomment to regenerate predictions.
# subprocess.run(predict_cmd, cwd=REPO_ROOT, check=True)
subprocess.run(check_cmd, cwd=REPO_ROOT, check=True)


## 4. Results and comparison with RVT

????? 200 ?????????

| Metric | Value |
|---|---:|
| Validation accuracy | **77.5%** |
| Validation macro-F1 | **79.4%** |
| Hidden test accuracy | **77.25%** |

Confusion matrix (`rows=true`, `columns=pred`):

```text
[[55, 21, 0], [15, 68, 8], [0, 1, 32]]
```

? RVT ?????RVT ?????/???????? 87.3% test R-Accuracy??????? classification-only pipeline????????????? 75% ????????? RVT?


In [ ]:
# Load final metrics from the local artifact.
metrics_path = REPO_ROOT / "outputs" / "convnext_tiny_final" / "valid_metrics.json"
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print(json.dumps(metrics, indent=2, ensure_ascii=False))


## 5. Report, GitHub link, and weight link

- GitHub ?????[https://github.com/wang935/homework](https://github.com/wang935/homework)
- GitHub ???????[https://github.com/wang935/homework/blob/main/models/final_model.pt](https://github.com/wang935/homework/blob/main/models/final_model.pt)
- GitHub ???????[https://github.com/wang935/homework/blob/main/outputs/results.csv](https://github.com/wang935/homework/blob/main/outputs/results.csv)
- ?? PDF ???[https://github.com/wang935/homework/blob/main/report/main_final.pdf](https://github.com/wang935/homework/blob/main/report/main_final.pdf)

> ??????? `models/final_model.pt` ?? Git LFS ???GitHub ??????? LFS ?????


## 6. Grading rubric checklist

- Code: refactored PyTorch code is in `src/` and `scripts/`; final `.pt` is uploaded.
- Method: includes CNN/model factory, preprocessing, weighted/class-balanced training, MixUp and augmentation.
- Results: validation accuracy 77.5%, hidden test accuracy 77.25%, above the 75% threshold.
- Discussion: report compares classification-only model with RVT and notes limitations.
